In [0]:
# ============================================================
# Bronze — Source 07: ShipStation API
#
# Incremental load using watermark pattern.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=07_shipstation/
# Target: bronze.shipstation catalog in Unity Catalog
#
# Tables:
#   - bronze.shipstation.shipments
#
# Raw format: Parquet (daily partitioned)
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "07_shipstation"

# Inspect raw files
dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/")

In [0]:
# ============================================================
# Incremental load for ShipStation shipments
#
# Raw format: JSON (one file per day, flat records not nested array)
# Join key: order_id → joins to bronze.postgres.orders
# ============================================================

watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/*.json"

df = spark.read.format("json") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

row_count = df.count()

if row_count == 0:
    print(f"[{SOURCE}] No new data — skipping")
else:
    print(f"Schema:")
    df.printSchema()
    df.show(3)

In [0]:
# Explode items array and flatten nested structs
# Each file has a wrapper with count, date, and items array
# order_id is the join key to bronze.postgres.orders

from pyspark.sql.functions import explode

shipments_flat = df \
    .select(explode(col("items")).alias("shipment")) \
    .select(
        col("shipment.shipment_id"),
        col("shipment.order_id"),
        col("shipment.carrier"),
        col("shipment.service"),
        col("shipment._source").alias("source"),
        col("shipment.created_at"),
        col("shipment.shipped_at"),
        col("shipment.delivered_at"),
        col("shipment.estimated_delivery"),
        col("shipment.label_url"),
        col("shipment.shipping_address.city").alias("shipping_city"),
        col("shipment.shipping_address.country").alias("shipping_country"),
    )

row_count = shipments_flat.count()
print(f"Flattened shipments: {row_count} rows")
shipments_flat.show(3)

In [0]:
# Cell 4 — Write to Bronze and update watermark
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.shipstation")

shipments_flat.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.shipstation.shipments")

latest_ts = df.select(spark_max("_metadata.file_modification_time")).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, row_count)
print(f"✅ bronze.shipstation.shipments: {row_count} rows loaded")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.shipstation.shipments").collect()[0]['cnt']
print(f"bronze.shipstation.shipments: {count} rows")
spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '07_shipstation'").show()